# Kvasir-VQA x1 — BLIP VQA baseline

Evaluate BLIP VQA (zero-shot) on Kvasir-VQA x1 metadata. Computes BLEU/ROUGE-L on a sample and yes/no accuracy on the yes/no subset.

In [25]:
import os
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from transformers import BlipProcessor, BlipForQuestionAnswering
from datasets import load_dataset
import evaluate


In [26]:
# Paths & config
HF_DATASET = "SimulaMet-HOST/Kvasir-VQA"  # adjust if using a different HF source
def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "blip_baseline" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Salesforce/blip-vqa-base"
SAMPLE_N = 500  # set None for all rows (can be slow)
SAMPLE_YN = 500  # yes/no subset sample
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)


Data root: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/blip_baseline/out
Device: cuda


In [27]:
# Load metadata
meta = pd.read_csv(META_CSV)
print("Rows:", len(meta))
print(meta.head())

# Use train/val/test if available else full set
def pick_split(df, split_name):
    if "split" in df.columns and split_name in df["split"].unique():
        return df[df["split"] == split_name].reset_index(drop=True)
    return df.reset_index(drop=True)

dev_df = pick_split(meta, "validation")
if len(dev_df) == 0:
    dev_df = pick_split(meta, "test")
if len(dev_df) == 0:
    dev_df = meta.copy()
print("Dev rows:", len(dev_df))


Rows: 58849
  split                     img_id  \
0   raw  cla820gl0s3nv071u4fgd7xgq   
1   raw  cla820gl0s3nv071u4fgd7xgq   
2   raw  cla820gl0s3nv071u4fgd7xgq   
3   raw  cla820gl0s3nv071u4fgd7xgq   
4   raw  cla820gl0s3nv071u4fgd7xgq   

                                     image_path  \
0  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   
1  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   
2  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   
3  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   
4  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   

                                            question              answer  \
0  Are there any abnormalities in the image? Chec...  ulcerative colitis   
1  Are there any anatomical landmarks in the imag...                none   
2  Are there any instruments in the image? Check ...                none   
3                      Have all polyps been removed?        not relevant   
4                    Is this finding easy to detect?                 yes

In [28]:
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForQuestionAnswering.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()
print("Loaded BLIP")


Loaded BLIP


In [29]:
# Fallback: load HF dataset and map img_id -> image if local file missing
HF_IMG_MAP = None
HF_DATASET_CACHE = None

def load_fallback_image(img_id: str):
    global HF_IMG_MAP, HF_DATASET_CACHE
    if HF_IMG_MAP is None:
        # try to load from HF
        try:
            ds_all = load_dataset(HF_DATASET)
            # pick first split
            if 'raw' in ds_all:
                ds_use = ds_all['raw']
            elif 'train' in ds_all:
                ds_use = ds_all['train']
            else:
                ds_use = list(ds_all.values())[0]
            HF_DATASET_CACHE = ds_use
            HF_IMG_MAP = {}
            for ex in ds_use:
                iid = ex.get('img_id') or ex.get('image_id') or ex.get('id') or ex.get('filename')
                if iid is None:
                    continue
                iid = str(iid).split('/')[-1].split('.')[0]
                HF_IMG_MAP[iid] = ex.get('image')
            print("Built HF image map:", len(HF_IMG_MAP))
        except Exception as e:
            print("Fallback HF load failed:", e)
            HF_IMG_MAP = {}
    return HF_IMG_MAP.get(str(img_id))


In [30]:
def generate_answer(row):
    img_path = Path(row["image_path"])
    img = None
    if img_path.is_file():
        img = Image.open(img_path).convert("RGB")
    else:
        img_id = row.get("img_id") or img_path.stem
        img = load_fallback_image(img_id)
        if img is None:
            raise FileNotFoundError(f"Image not found locally and HF fallback missing for {img_path}")
        if img.mode != "RGB":
            img = img.convert("RGB")
    q = str(row["question"])
    inputs = processor(images=img, text=q, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20)
    ans = processor.decode(out[0], skip_special_tokens=True)
    return ans

# BLEU/ROUGE metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

def eval_sample(df, n=None):
    if n is not None:
        df = df.sample(min(n, len(df)), random_state=SEED).reset_index(drop=True)
    preds = []
    refs = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="BLIP eval"):
        ans = generate_answer(row)
        preds.append(ans)
        refs.append(str(row["answer"]))
    bleu_refs = [[r] for r in refs]  # sacrebleu expects list-of-list references
    bleu_score = bleu.compute(predictions=preds, references=bleu_refs)["bleu"]
    rouge_l = rouge.compute(predictions=preds, references=refs)["rougeL"].mid.fmeasure
    return preds, refs, bleu_score, rouge_l


In [31]:
# Run eval on a sample for BLEU/ROUGE
preds, refs, bleu_score, rouge_l = eval_sample(dev_df, n=SAMPLE_N)

dev_df_sampled = dev_df.head(len(preds)).copy()
dev_df_sampled["pred_blip"] = preds

print("BLEU:", bleu_score)
print("ROUGE-L:", rouge_l)

# Save predictions
pred_path = OUT_DIR / "predictions_blip_sample.csv"
dev_df_sampled.to_csv(pred_path, index=False)
print("Saved:", pred_path)


BLIP eval:   0%|          | 0/500 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/30 [00:00<?, ?it/s]

Built HF image map: 6500


AttributeError: 'numpy.float64' object has no attribute 'mid'

In [ ]:
# Yes/No subset accuracy
yn_df = dev_df[dev_df["answer"].astype(str).str.lower().isin(["yes","no"])]
if SAMPLE_YN is not None:
    yn_df = yn_df.sample(min(SAMPLE_YN, len(yn_df)), random_state=SEED).reset_index(drop=True)

def to_yesno(text):
    t = str(text).lower()
    if "yes" in t:
        return "yes"
    if "no" in t:
        return "no"
    return "no"

yn_preds = []
for _, row in tqdm(yn_df.iterrows(), total=len(yn_df), desc="BLIP yes/no"):
    ans = generate_answer(row)
    yn_preds.append(to_yesno(ans))

y_true = yn_df["answer"].str.lower().tolist()
acc_yn = (pd.Series(yn_preds) == pd.Series(y_true)).mean() if len(yn_df) else 0
print("Yes/No accuracy:", acc_yn, "(n=", len(yn_df), ")")

yn_df_out = yn_df.copy()
yn_df_out["pred_blip_yesno"] = yn_preds
yn_path = OUT_DIR / "predictions_blip_yesno.csv"
yn_df_out.to_csv(yn_path, index=False)
print("Saved yes/no preds:", yn_path)


## Notes
- Adjust `SAMPLE_N` / `SAMPLE_YN` to control runtime (set to None for full eval).
- BLEU/ROUGE are on the sampled set; yes/no accuracy is on the yes/no subset.
- Use results to decide if fine-tuning or constrained decoding is needed.
